In [5]:
# -*- coding: utf-8 -*-
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import random
from torch.utils.data import DataLoader

In [6]:
# ===== convolution kernel =====
custom_kernel = torch.zeros((32, 1, 3, 3))
custom_kernels = torch.stack([
    torch.tensor([[0, 1, 0],
                  [0, 1, 0],
                  [0, 1, 0]], dtype=torch.float32),
    torch.tensor([[1, 0, 0],
                  [0, 1, 0],
                  [0, 0, 1]], dtype=torch.float32),
    torch.tensor([[0, 1, 0],
                  [1, 1, 1],
                  [0, 1, 0]], dtype=torch.float32)
]).unsqueeze(1)

# ===== CNN model =====
class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=3, kernel_size=3, padding=1, bias=False)
        self.conv1.weight = nn.Parameter(custom_kernels, requires_grad=False)
        self.conv2 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 14 * 14, 10)


    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 32 * 14 * 14)
        x = torch.relu(self.fc1(x))

        return x

In [7]:
# ===== choose dataset =====
# dataset：'MNIST', 'FashionMNIST', 'KMNIST'
dataset_name = 'FashionMNIST'

# ===== load dataset =====
def load_dataset(name):
    transform = transforms.Compose([
        transforms.Grayscale(),  # Convert colors to grayscale
        transforms.ToTensor(),
    ])
    if name == 'MNIST':
        dataset = torchvision.datasets.MNIST
    elif name == 'FashionMNIST':
        dataset = torchvision.datasets.FashionMNIST
    elif name == 'KMNIST':
        dataset = torchvision.datasets.KMNIST
    else:
        raise ValueError("dataset loading error！")

    train_data = dataset(root='./data', train=True, transform=transform, download=False)
    test_data = dataset(root='./data', train=False, transform=transform, download=False)

    return train_data, test_data

In [ ]:
# ===== train_and_test =====
def train_and_test():
    train_dataset, test_dataset = load_dataset(dataset_name)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = CustomCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(10):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

    # accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"accuracy: {100 * correct / total:.2f}%")

# ===== main function =====
if __name__ == '__main__':
    train_and_test()

Epoch 1, Loss: 1.6857
Epoch 2, Loss: 1.2986
Epoch 3, Loss: 1.2649
accuracy: 58.06%
